# 문제해결 1: 금융사기 보고서 기반 FAISS Naive RAG

## 실습 목적

Naive RAG의 기본 흐름을 익힌 후, **직접 문서 기반 질의응답 시스템을 구성하고 문제를 점검**하도록 설계한 실습 예제

사용 문서: `data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf`

## 실습 시나리오

금융소비자 교육 담당자가 하나의 금융사기 보고서를 기반으로 질문에 답변하는 내부용 QA 챗봇을 만들려고 한다.  

## 완성 기준

| 기준 | 설명 |
|---|---|
| 문서 로딩 | PDF 페이지를 `Document` 객체로 정상 로딩한다. |
| 청킹 | 문서가 적절한 크기의 청크로 분할된다. |
| 벡터DB 생성 | FAISS 벡터스토어가 생성되고 로컬 저장까지 가능하다. |
| 검색 | 질문과 관련된 상위 문서 조각을 검색한다. |
| 답변 생성 | 검색된 문서에 근거해서만 답변한다. |

## 0. 설치

> `faiss-cpu`는 로컬 CPU 환경에서 FAISS를 사용하기 위한 패키지이다.

In [1]:
# uv add langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv pandas

## 1. 라이브러리 불러오기

LangChain의 기본 구성요소를 사용한다.

- `PyPDFLoader`: PDF 문서 로딩
- `RecursiveCharacterTextSplitter`: 문서 청킹
- `OpenAIEmbeddings`: 텍스트 임베딩
- `FAISS`: 로컬 벡터스토어
- `ChatOpenAI`: 답변 생성 LLM

In [2]:
from pathlib import Path
import os
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_3124473/1765998713.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 2. 환경변수 설정

OpenAI API Key는 `.env` 파일 또는 운영체제 환경변수에 설정한다.

`.env` 파일 예시는 다음과 같다.

```text
OPENAI_API_KEY=sk-...
```

In [3]:
load_dotenv(override=True, dotenv_path="../.env")

if not os.getenv("LLM_API_KEY"):
    raise ValueError(
        "LLM_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 LLM_API_KEY를 설정하세요."
    )

print("LLM_API_KEY 설정 확인 완료")

LLM_API_KEY 설정 확인 완료


## 3. PDF 파일 경로 설정

노트북과 같은 폴더 또는 `data` 폴더에 PDF를 넣어두면 자동으로 찾는다.

권장 폴더 구조는 다음과 같다.

```text
project/
├─ 1_faiss_naive_rag_financial_fraud.ipynb
└─ data/
   └─ 진화하는 금융사기, 모두의 경계가 필요한 시점.pdf
```

In [4]:
PDF_NAME = "진화하는 금융사기, 모두의 경계가 필요한 시점.pdf"

candidate_paths = [
    Path(PDF_NAME),
    Path("data") / PDF_NAME,
    Path("/mnt/data") / PDF_NAME,  # ChatGPT 작업환경용 경로
]

PDF_PATH = next((path for path in candidate_paths if path.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError(
        f"PDF 파일을 찾을 수 없습니다. 다음 위치 중 하나에 파일을 넣어주세요: {candidate_paths}"
    )

print("PDF_PATH:", PDF_PATH)

PDF_PATH: data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 4. PDF 로딩

PDF를 페이지 단위 `Document`로 로딩한다.  
각 페이지는 `page_content`와 `metadata`를 가진다.

### pdf 파일 로딩

In [5]:
# PDF 로드
loader = PyPDFLoader(str(PDF_PATH))

# PDF 페이지를 로드하여 문서 객체 리스트로 반환
pages = loader.load()

# 메타데이터 보강
for doc in pages:
    doc.metadata["source_name"] = PDF_NAME
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1

print("로드된 페이지 수:", len(pages))
print("첫 번째 페이지 메타데이터:")
pprint(pages[0].metadata)
print("\n첫 번째 페이지 내용 일부:")
print(pages[0].page_content[:500])

로드된 페이지 수: 15
첫 번째 페이지 메타데이터:
{'author': 'user',
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 페이지 내용 일부:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 5. 문서 청킹

RAG에서는 문서 전체를 한 번에 넣지 않고, 검색 가능한 단위로 나눈다.  
 RecursiveCharacterTextSplitter()을 사용한다.  
이 실습에서는 보고서형 PDF에 적합한 기본값으로 `chunk_size=800`, `chunk_overlap=120`을 사용한다.

- `chunk_size`: 하나의 문서 조각 최대 길이
- `chunk_overlap`: 앞뒤 청크가 겹치는 길이
- 목적: 문맥 손실을 줄이면서 검색 단위를 적절히 유지하는 것

In [6]:
# 텍스트 분할 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 청크 크기
    chunk_overlap=200,  # 청크 겹침
    length_function=len,
)

# 페이지 단위로 로드된 문서를 청크 단위로 분할
chunks = text_splitter.split_documents(pages)

# 청크 단위로 메타데이터 보강
for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    doc.metadata["source_name"] = PDF_NAME

print("전체 청크 수:", len(chunks))
print("첫 번째 청크 메타데이터:")
pprint(chunks[0].metadata)
print("\n첫 번째 청크 내용:")
print(chunks[0].page_content[:800])

전체 청크 수: 23
첫 번째 청크 메타데이터:
{'author': 'user',
 'chunk_id': 0,
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'pdfversion': '1.4',
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'total_pages': 15}

첫 번째 청크 내용:
2026년 5월 29일 제 17 호
진화하는 금융사기,
모두의 경계가 필요한 시점


## 6. 청크 분포 확인

페이지별 청크 수를 확인한다.  
특정 페이지의 청크가 지나치게 많거나 적으면 청킹 전략을 조정해야 한다.

In [7]:
# 청크 정보를 DataFrame으로 정리하여 페이지별 청크 수와 샘플을 확인
chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata.get("page_label"),
        "text_length": len(doc.page_content),
        "preview": doc.page_content[:80].replace("\n", " ")
    }
    for doc in chunks
])

print("페이지별 청크 수")
display(chunk_df.groupby("page").size().reset_index(name="chunk_count"))

print("청크 샘플")
display(chunk_df.head(10))

페이지별 청크 수


,page,chunk_count
0,1,1
1,2,2
2,3,1
3,4,2
4,5,2
5,6,1
6,7,2
7,8,2
8,9,1
9,10,2


청크 샘플


,chunk_id,page,text_length,preview
0,0,1,45,"2026년 5월 29일 제 17 호 진화하는 금융사기, 모두의 경계가 필요한 시점"
1,1,2,964,하나은행 ...
2,2,2,275,"∎ 금융사기로 발생하는 금전적 피해 규모는 증가 추세이며, 사기 수법이 고도화되고..."
3,3,3,926,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...
4,4,4,996,하나은행 ...
5,5,4,288,표1 | 금융사기 발생 흐름 그림1 | 금융사기(핀플루언서) 사례 단계 내용 신뢰 ...
6,6,5,995,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
7,7,5,303,메신저피싱 카카오톡이나 SNS로 타인을 사칭하여 송금을 유도하는 수법 파밍 가짜 ...
8,8,6,934,하나은행 ...
9,9,7,973,Hana Bank Hana Institute of Finance 6 III. 금융사...


## 7. 임베딩 모델과 FAISS 벡터스토어 생성

여기서부터 인덱싱 단계이다.

```text
청크 텍스트 → 임베딩 벡터 → FAISS 인덱스 저장
```

FAISS는 벡터 간 유사도 검색을 빠르게 수행하기 위한 로컬 벡터 검색 라이브러리이다.  
이 노트북에서는 LangChain의 `FAISS.from_documents()`를 사용해 문서 청크와 임베딩 모델을 연결한다.

In [8]:
# 1. 임베딩 모델 초기화
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=os.getenv("LLM_API_KEY"), base_url=os.getenv("LLM_BASE_URL"))

# 2. FAISS 인덱스 저장 경로 설정
INDEX_DIR = Path("faiss_financial_fraud_pdf_index")
FAISS_INDEX_FILE = INDEX_DIR / "index.faiss"
FAISS_PKL_FILE = INDEX_DIR / "index.pkl"

# 3. 기존 인덱스가 있으면 로딩, 없으면 새로 생성
if FAISS_INDEX_FILE.exists() and FAISS_PKL_FILE.exists():
    print("기존 FAISS 인덱스를 로딩합니다.")

    # FAISS 벡터스토어 로딩 
    vectorstore = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)

    print("FAISS 인덱스 로딩 완료:", INDEX_DIR.resolve())

else:
    print("기존 FAISS 인덱스가 없습니다.")
    print("청크 문서를 임베딩하여 새 FAISS 벡터스토어를 생성합니다.")

    # FAISS 벡터스토어 생성
    vectorstore = FAISS.from_documents(chunks, embeddings)

    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(INDEX_DIR))

    print("FAISS 인덱스 생성 및 저장 완료:", INDEX_DIR.resolve())

# 4. 인덱스 상태 확인
print("저장된 문서 청크 수:", vectorstore.index.ntotal)


기존 FAISS 인덱스를 로딩합니다.


ValueError: The de-serialization relies loading a pickle file. Pickle files can be modified to deliver a malicious payload that results in execution of arbitrary code on your machine.You will need to set `allow_dangerous_deserialization` to `True` to enable deserialization. If you do this, make sure that you trust the source of the data. For example, if you are loading a file that you created, and know that no one else has modified the file, then this is safe to do. Do not set this to `True` if you are loading a file from an untrusted source (e.g., some random site on the internet.).

## 8. 기본 유사도 검색 테스트

먼저 LLM 답변 생성 없이, 검색 결과만 확인한다.  
이 단계가 중요하다. 검색이 잘못되면 답변도 잘못될 가능성이 높다.

`similarity_search_with_score()`의 score는 기본 FAISS 설정에서는 거리 기반 점수로 해석한다.  
일반적으로 **낮을수록 더 가까운 문서**로 볼 수 있다.

In [ ]:
question = "금융사기는 어떤 단계로 발생하나요?"

results = vectorstore.similarity_search_with_score(question, k=5)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace("\n", " "))


[1] score=0.8507 | page=2 | chunk=2
∎ 금융사기로 발생하는 금전적 피해 규모는 증가 추세이며, 사기 수법이  고도화되고 복잡해지면서 금융사기 예방 효과는 여전히 부족한 상황 l 보이스피싱, 유사수신 등 불특정 다수를 대상으로 이루어지는 금융사기로 인한  피해금액은 ’24년 약 1.7조 원으로 큰 폭으로 증가하였으며 지속적 증가 예상  l 과거 금융사기는 기술 활용이 익숙하지 않은 고령층을 중심으로 발생하는 것으로  여겨졌으나, 수법이 다양하고 복잡해지면서 피해 연령층도 다양화되는 경향 < Executive Summary >

[2] score=0.8539 | page=4 | chunk=5
표1 | 금융사기 발생 흐름 그림1 | 금융사기(핀플루언서) 사례 단계 내용 신뢰 유도  금융기관, 공공기관, 지인 등을 사칭하여  피해자의 심리적 경계를 모호하게 만드는 단계로  피해자와의 사적 관계를 구축 개인정보  수집  개인정보나 금융정보를 탈취하는 단계로  신분증, 통장사본 등을 요구하거나 앱을  설치하도록 하는 등 송금이 가능한 경로를 생성 금전 편취  및 잠적  피해자의 계좌에서 자금을 인출하거나 대출을  실행하고 연락이 두절되어 피해자가  금전적 피해를 입게 됨 자료 : 경찰청 자료 : 금융감독원

[3] score=0.8809 | page=13 | chunk=20
사회적 문제 해결을 위한 공유 체계를 적극적으로 마련하여 피해를 예방하고자 노력 ■ 금융사기는 개인이 순간적으로 사기를 인지하고 예방하기 어렵다는 측면에서  금융회사와 관련 정부 기관의 예방 노력이 함께 이행되어야 함 ● 개별법 제·개정, 규제 강화 중심의 예방 노력은 후행적 조치라는 한계가 존재하므로   금융사기 예방을 위한 범기관 네트워크 구축이 요구 - 통신사, 가상자산거래소, 은행 등 금융사기가 발생하는 지점과 직접적으로 관련된 기관과의  공유 네트워크를 구축함으로써 민관 협력을 통한 예방 체계를 고도화 ● 금융사는 금융사기 전담 부서 조직을 확충하고 금융사기를 

## 9. Retriever 구성

Retriever는 사용자의 질문을 받아 관련 문서 조각을 반환하는 객체이다.

```text
질문 → Retriever → 관련 청크 Top-k 반환
```

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

retrieved_docs = retriever.invoke("전기통신금융사기와 투자사기의 차이는 무엇인가요?")

for doc in retrieved_docs:
    print(f"page={doc.metadata.get('page_label')}, chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:300].replace("\n", " "))
    print("-" * 80)

page=7, chunk=10
경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입 표3 | 전기통신금융사기 주요 대응 현황 표4 | 투자사기 주요 대응 현황 시기 주요 내용 시기 주요 내용 ’23.7 통합신고대응센터 개소 ’24.1 자본시장법  개정  Ÿ 불공정거래 부당이득의 최대 2배   과징금 항목 신설’24.1 은행권 자율배상제도 시행 ’24.8 여신거래 안심차단 서비스 시행 ’24.7 가상자산이용자 보호법 시행 Ÿ 미공개정보이용·시세조종·사기적  거래행위 처벌 강화’25.3 비대면 계좌개설 안심차단 서비스 시행 ’24.8 자본시장
--------------------------------------------------------------------------------
page=5, chunk=6
Hana Bank Hana Institute of Finance 4 2. 금융사기 주요 유형 ■ 금융사기는 금전적 피해를 유발하는 범죄행위 전반을 포괄해 다양한 기준으로  구분되나 전기통신금융사기와 투자사기 두 분류로 구분하는 것이 보편적 ● 전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용하여 금융기관·    지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 수법 - 피싱, 스미싱, 파밍 등이 대표적인 유형이며, 법에서 열거하는 통신수단을 이용하여   개인정보를 노출시켜 자금을 송금·이체하도록 유도 - ’
--------------------------------------------------------------------------------
page=2, chunk=2
∎ 금융사기로 발생하는 금전적 피해 규모는 증가 추세이며, 사기 수법이  고도화되고 복잡해지면서 금융사기 예방 효과는 여전히 부족한 상황 l 보이스피싱, 유사수신 등 불특정 다수를 대상으로 이루어지는 금융사기로 인한  피해금액은 ’24년 약 1.7조 원으로 큰 폭으로 증가하였으며 지속적 증가 예상  l 과거 금융사기는 기술 활용이 익

## 10. RAG 답변 프롬프트 구성

핵심은 LLM이 검색된 문서 밖의 내용을 임의로 생성하지 않도록 제약하는 것이다.

프롬프트 설계 원칙은 다음과 같다.

1. 문서 근거만 사용한다.
2. 문서에서 확인할 수 없으면 모른다고 답한다.
3. 답변 마지막에 근거 페이지를 포함한다.

In [ ]:
# 검색 chunk를 context로 변환하는 함수
def format_docs(docs):
    """검색된 문서들을 LLM 입력용 context 문자열로 변환한다."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page_label", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        formatted.append(
            f"[출처: page {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


# 4. LLM과 프롬프트 초기화
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.
아래 [문서]에 근거해서만 답변하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.
답변은 핵심 위주로 3~5문장으로 정리하세요.
답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip()
    ),

    ("human", "{question}"),
])

# model = gpt-4o-mini로 설정, temperature=0으로 설정
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0, openai_api_key=os.getenv("LLM_API_KEY"), base_url=os.getenv("LLM_BASE_URL"))

# 5. LCEL Chain 구성(프롬프트, LLM, 문자열 출력 파서)
answer_chain = prompt | llm | StrOutputParser()

In [ ]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

# 질문에 대한 유사도 top 4 검색 하기
docs = retriever.invoke(question, k=4)

# 검색된 문서를 LLM 입력용 context로 변환
context = format_docs(docs)

# LLM에 질문과 검색된 문서 컨텍스트를 입력하여 답변 생성
answer = answer_chain.invoke({"context": context, "question": question})

In [ ]:
# 결과 출력
answer

'전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용해 금융기관·지인·가족을 사칭하거나 허위 사실로 피해자의 자금을 송금·이체하게 만들어 탈취하는 방식입니다. 반면 투자사기는 투자자의 수익 기대를 이용해 투자를 유도하거나, 인위적 가격 조작·허위 수익률 제시 등으로 금전적 피해를 유발하는 범죄입니다. 문서상 전기통신금융사기의 대표 유형은 보이스피싱, 메신저피싱, 파밍, 스미싱, 해킹이며, 투자사기는 리딩방 사기, 비상장주식 과장 판매, 가짜 HTS/MTS 유도가 예시로 제시됩니다. 즉, 전자는 통신수단을 이용한 자금 편취가 중심이고, 후자는 투자 행위와 수익을 미끼로 한 사기가 중심입니다. 근거: p.4, p.5'

## 11. RAG 함수 만들기

검색과 답변 생성을 하나의 함수로 묶는다.

```text
질문 입력
→ FAISS 검색
→ 검색 문서 format
→ LLM 답변 생성
→ 답변과 출처 반환
```

In [ ]:
def answer_question(question: str, k: int = 4):
    """질문에 대해 FAISS 검색 후 문서 기반 답변과 출처 테이블을 반환한다."""
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    answer = answer_chain.invoke({
        "question": question,
        "context": context,
    })

    sources = pd.DataFrame([
        {
            "rank": i + 1,
            "page": doc.metadata.get("page_label"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "preview": doc.page_content[:120].replace("\n", " ")
        }
        for i, doc in enumerate(docs)
    ])

    return answer, sources

## 12. RAG 실행 예시

다음 질문들은 이 보고서에 포함된 내용을 기반으로 답변 가능한 질문이다.

In [ ]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)

print("\n[검색된 근거]")
display(sources)  # Jupyter 환경에서는 display로 테이블이 예쁘게 나옵니다.


[질문]
전기통신금융사기와 투자사기는 어떻게 다른가요?

[답변]
전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용해 금융기관·지인·가족 등을 사칭하거나 허위 사실로 피해자를 속여 자금을 송금·이체하게 만드는 사기입니다. 반면 투자사기는 투자자의 수익 기대를 악용해 투자를 유도하거나 인위적으로 가격을 조작해 금전적 피해를 유발하는 사기입니다. 전기통신금융사기는 보이스피싱·메신저피싱·파밍·스미싱·해킹 등이 대표적이고, 투자사기는 리딩방 사기·비상장 주식 과장 판매·가짜 HTS/MTS 유도가 대표적입니다. 즉, 전기통신금융사기는 “통신수단을 이용한 기망”, 투자사기는 “투자 유도나 시세 조작을 통한 피해”가 핵심 차이입니다. 근거: p.4, p.5

[검색된 근거]


,rank,page,chunk_id,preview
0,1,7,10,경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입 표3 ...
1,2,5,6,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
2,3,3,3,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...
3,4,7,9,Hana Bank Hana Institute of Finance 6 III. 금융사...


In [ ]:
question = "금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?

[답변]
금융회사들은 전자금융거래 패턴을 실시간 분석해 이상거래를 탐지·차단하는 이상금융거래탐지시스템(FDS)을 도입하고 있습니다. 또한 금융사기 전담 조직을 확충하고, 자체 사기 방지 소프트웨어를 개발하는 등 내부 통제 체계를 강화하고 있습니다. 더 나아가 AI를 결합한 탐지 시스템과 생성형 AI를 활용해 신종 사기 수법에 선제 대응하며, 통신사 등 유관기관과 실시간 협업 체계도 구축하고 있습니다. 예시로 JPMorgan Chase의 ‘NeuroShield’, Commonwealth Bank의 ‘Scam Indicator’와 ‘NameCheck’가 언급됩니다. 근거: p.9, p.12

[검색된 근거]


,rank,page,chunk_id,preview
0,1,12,17,하나은행 ...
1,2,9,13,Hana Bank Hana Institute of Finance 8 2. 금융회사 ...
2,3,12,18,금융사기를 사전에 차단할 수 있는 시스템을 자체적으로 구축 ● 미국 JPMorgan...
3,4,13,20,사회적 문제 해결을 위한 공유 체계를 적극적으로 마련하여 피해를 예방하고자 노력 ■...


## 13. 문서에 없는 질문 테스트

RAG 실습에서는 답변 가능한 질문만 넣으면 안 된다.  
문서에 없는 질문을 넣어 **모른다고 답하는지** 확인해야 한다.

In [ ]:
question = "2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?

[답변]
문서에 따르면 보이스피싱·유사수신 등 불특정 다수를 대상으로 한 금융사기 피해금액은 ’24년 약 1.7조 원으로 큰 폭으로 증가한 것으로 제시되어 있습니다. 또한 로맨스스캠은 ’25년 1,360억 원, 투자사기는 ’25년 월평균 548억 원으로 언급되어 있으나, **2026년 6월 기준 최신 보이스피싱 피해 금액**은 문서에서 확인할 수 없습니다. 따라서 요청하신 시점의 최신 수치는 문서에서 확인할 수 없습니다.  
근거: p.6

[검색된 근거]


,rank,page,chunk_id,preview
0,1,10,15,"받을 수 있게 되며 금융회사가 부담해야 할 배상액 규모는 2,811억 원으로 추정 ..."
1,2,10,14,하나은행 ...
2,3,6,8,하나은행 ...
3,4,8,12,정보를 실시간으로 공유하고 범죄 예방에 활용 ● 금융보안원은 금융사기 관련 정보를...


## 14. 정리

이 실습에서 구현한 Naive RAG 흐름은 다음과 같다.

```text
PDF 로딩
→ 페이지 단위 Document 생성
→ RecursiveCharacterTextSplitter로 청킹
→ OpenAIEmbeddings로 임베딩
→ FAISS 벡터스토어 생성
→ 질문 기반 Top-k 검색
→ 검색 문서 기반 LLM 답변 생성
```

## 핵심 포인

| 관찰 포인트 | 핵심 내용 |
|---|---|
| 검색 결과가 부정확함 | 청크 크기, 질문 표현, Top-k 조정 필요 |
| 답변이 문서 밖 내용을 포함함 | 프롬프트 제약 및 문서 없음 질문 테스트 필요 |
| 표·그림 내용이 누락됨 | PDF 파싱 방식의 한계, Markdown 변환 비교 필요 |
| 같은 질문인데 답변이 달라짐 | 검색 결과와 LLM 생성 조건을 분리해서 점검해야 함 |
| 인덱스 재사용 필요 | FAISS 저장·로드 구조로 인덱싱 비용 절감 가능 |